In [26]:
# библиотеки и загрузка данных
import pandas as pd
from scipy.stats import f # f-распределение и критические значения
from scipy.stats import t # t-распределение и критические значения
import numpy as np
from statsmodels.formula.api import ols

# импорт данных
df = pd.read_csv('sleep75.csv')


In [27]:
# построение регресионной модели

mod = ols(formula='sleep ~ totwrk +age +south +male + smsa', data=df)
res = mod.fit() #  неробастные ошибки, если указать res1 = mod1.fit(cov_type='HC3') будет модель с робастными ошибками

print(res.summary()) #распечатать полное summary

                            OLS Regression Results                            
Dep. Variable:                  sleep   R-squared:                       0.131
Model:                            OLS   Adj. R-squared:                  0.124
Method:                 Least Squares   F-statistic:                     21.02
Date:                Fri, 05 Jun 2026   Prob (F-statistic):           1.32e-19
Time:                        09:02:03   Log-Likelihood:                -5256.2
No. Observations:                 706   AIC:                         1.052e+04
Df Residuals:                     700   BIC:                         1.055e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3470.4602     69.377     50.023      0.0

In [ ]:
# sleep75 ~ log(faminc) +educ + selfe + yrsmarr  + age

In [ ]:
#################### интерпретация коэф-тов ####################
#1. линейная модель y ~ x
# b1: при увеличении x1 на 1 усл. ед. при прочих равных  y в среднем  увеличится/уменьшится (+/-) на b1 усл. ед.

#2. лог-линейная модель: log(y) ~ x
# b1: при увеличении x на единицу при прочих равных y в среднем изменится на b1*100%

#3. лог-лог модель: log(y) ~ log(x)
# b1: При увеличении x на 1% при прочих равных ,зависимая переменная в среднем изменится на 1%

#4. лин-лог модель: y ~ log(x)
# b1: При увеличении x на 1% при прочих равных, зависимая переменная y в среднем изменится на b1/100

In [ ]:
#################### прогнозы ####################
#newdata -  датасет, для которого считаем прогноз

new_data = df.loc[12] #записываем в датасет значение из начального датафрейма с индексом 12
print(new_data.round(3)) #фактические значения


age           24.000
black          0.000
case          13.000
clerical       0.000
construc       0.000
educ          16.000
earns74     1000.000
gdhlth         1.000
inlf           1.000
leis1       3844.000
leis2       3844.000
leis3       3794.000
smsa           1.000
lhrwage        0.207
lothinc        7.314
male           1.000
marr           1.000
prot           0.000
rlxall      3848.000
selfe          0.000
sleep       3798.000
slpnaps     3798.000
south          0.000
spsepay     3000.000
spwrk75        1.000
totwrk      2438.000
union          0.000
worknrm     2438.000
workscnd       0.000
exper          2.000
yngkid         0.000
yrsmarr        4.000
hrwage         1.230
agesq        576.000
Name: 12, dtype: float64


In [ ]:
#расчет прогнозных значений  - точечный прогноз
pred = res.predict(exog=new_data, transform=True)
print(pred.round(3))

12    3157.915
dtype: float64


In [ ]:
#ошибка прогноза
resid = new_data['sleep']-pred
print(resid.round(3))

12    640.085
dtype: float64


In [29]:
#способ 2 - вручную указать датасет прогнозных значений
new_data = pd.DataFrame({'totwrk': [2438], 'age': [24], 'south':[0], 'male':[1], 'smsa':[1]})
new_data

,totwrk,age,south,male,smsa
0,2438,24,0,1,1


In [30]:
#расчет прогнозных значений
pred = res.predict(exog=new_data, transform=True)
print(pred.round(3))

0    3157.915
dtype: float64


In [ ]:
################# значимость модели - тест Фишера - F тест ###################

In [ ]:
#предварительные шаги: загружаем данные + строим модель

In [31]:
#F-тест (неробастный)
F_test = res.f_test('totwrk =0, age=0, south=0, male=0, smsa=0')
#если проверяем совместную значимость, то вместо всех коэф модели указываем только проверяемые

print(F_test) #F - тестовая статистика, p - p-value

<F test: F=21.016710394068472, p=1.3201157713325485e-19, df_denom=700, df_num=5>


In [32]:
# уровень значимости - берем из условия
sign_level = 0.01
# Критическое значение F-распределения
f.ppf(q=1-sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

np.float64(3.043428128183389)

In [ ]:
# вывод:  F > F крит -> отвергаем H_0 -> модель значима (выбранные коэф значимы) на уровне значимости alpha

In [ ]:
#F-тест (робастный) - нужно предварительно построить модель с робастными ошибками

In [ ]:
mod = ols(formula='sleep ~ totwrk +age +south +male + smsa', data=df)
res = mod.fit(cov_type='HC3')

In [ ]:
F_test = res.f_test('totwrk =0, age=0, south=0, male=0, smsa=0')
#если проверяем совместную значимость, то вместо всех коэф модели указываем только проверяемые

print(F_test) #F - тестовая статистика, p - p-value

<F test: F=16.62122720403213, p=1.57086054797039e-15, df_denom=700, df_num=5>


In [33]:
sign_level = 0.01
# Критическое значение F-распределения
f.ppf(q=1-sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

np.float64(3.043428128183389)

In [ ]:
# вывод:  F > F крит -> отвергаем H_0 -> модель значима (выбранные коэф значимы) на уровне значимости alpha

In [ ]:
################# значимость  коэф - тест Стьюдента - t тест ###################

In [ ]:
#предварительные шаги: загружаем данные + строим модель

In [34]:
mod = ols(formula='sleep ~ totwrk +age +south +male + smsa', data=df)
res = mod.fit() #если нужны робастные ошибки -  res = mod.fit(cov_type='HC3')

In [35]:
# табличные значения
sign_level = 0.05
# критическое значение t-распределения
t.isf(q=sign_level/2, df=res.df_resid) #res1 - название модели

np.float64(1.9633587110998145)

In [37]:
#Но : Beta = 0. неробастный тест
res.t_test('totwrk = 0')

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.1702      0.018     -9.493      0.000      -0.205      -0.135

In [38]:
#Но : Beta = A. неробастный тест # t = критического значение , которое сравниваем с табличным
res.t_test('totwrk = - 0.3')

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.1702      0.018      7.238      0.000      -0.205      -0.135

In [39]:
#оценим модель с робастными ошибками
mod1 = ols(formula='sleep ~ totwrk +age +south +male + smsa', data=df)
res2 = mod1.fit(cov_type='HC3')
print(res2.summary())

                            OLS Regression Results                            
Dep. Variable:                  sleep   R-squared:                       0.131
Model:                            OLS   Adj. R-squared:                  0.124
Method:                 Least Squares   F-statistic:                     16.62
Date:                Fri, 05 Jun 2026   Prob (F-statistic):           1.57e-15
Time:                        09:03:17   Log-Likelihood:                -5256.2
No. Observations:                 706   AIC:                         1.052e+04
Df Residuals:                     700   BIC:                         1.055e+04
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3470.4602     74.783     46.407      0.0

In [43]:
#Но : Beta = 0. робастный тест # z = критическое значение , которое сравниваем с табличным
res2.t_test('totwrk = 0')

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.1702      0.020     -8.383      0.000      -0.210      -0.130

In [44]:
#Но : Beta = A. робастный тест # z = критическое значение , которое сравниваем с табличным
res2.t_test('totwrk = - 0.3')

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.1702      0.020      6.391      0.000      -0.210      -0.130

In [ ]:
# вывод: t (z) > t_табл (6.391 > 1.96) --> коэф beta_totwrk значительно отличается от -0.3

In [ ]:
# Если набл значение < критического, например t (z) =1.05 < t_табл (1.96) --> коэф beta_totwrk не отличается от -0.3

In [45]:
# регрессия с квадратом
# Для датасета wage1 рассмотрим регрессию log(wage) ~ exper + I(exper** 2) +married + smsa +services.

In [46]:
import pandas as pd
from scipy.stats import f # f-распределение и критические значения
import numpy as np
from statsmodels.formula.api import ols

# импорт данных
df = pd.read_csv('wage1.csv') #df - название датасета

In [50]:

mod1 = ols(formula='np.log(wage) ~ exper + I(exper**2) +married + smsa +services', data = df)
# подгонка исходной модели
res2 = mod1.fit()
print(res2.summary())


                            OLS Regression Results                            
Dep. Variable:           np.log(wage)   R-squared:                       0.210
Model:                            OLS   Adj. R-squared:                  0.203
Method:                 Least Squares   F-statistic:                     27.71
Date:                Fri, 05 Jun 2026   Prob (F-statistic):           6.62e-25
Time:                        09:06:00   Log-Likelihood:                -351.31
No. Observations:                 526   AIC:                             714.6
Df Residuals:                     520   BIC:                             740.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         1.0761      0.061     17.706

In [ ]:
################# DW-тест: тестируем автокорреляцию первого порядка, - тест Дарбина Уотсона - DW ###################

In [ ]:
# библиотеки и загрузка данных
import pandas as pd
from scipy.stats import f # f-распределение и критические значения
from scipy.stats import t # t-распределение и критические значения
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.api import durbin_watson, acorr_breusch_godfrey # DW & LM-тесты
from statsmodels.iolib.summary2 import summary_col # вывод подгонки

from scipy.stats import chi2 # chi2-распределение и критические значения

# импорт данных
df = pd.read_csv('Tbrate.csv')

In [ ]:
# спецификация модели
mod1 = ols(formula='pi  ~ r+ y', data = df)
# подгонка исходной модели
res = mod1.fit()

print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                     pi   R-squared:                       0.324
Model:                            OLS   Adj. R-squared:                  0.317
Method:                 Least Squares   F-statistic:                     44.35
Date:                Fri, 05 Jun 2026   Prob (F-statistic):           1.85e-16
Time:                        06:13:25   Log-Likelihood:                -457.84
No. Observations:                 188   AIC:                             921.7
Df Residuals:                     185   BIC:                             931.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     22.7725      6.476      3.516      0.0

In [ ]:
# DW-статистика
durbin_watson(res.resid)

np.float64(0.5085540848937082)

In [ ]:
#################  серийная корреляция второго или произвольного порядка - LM тест

In [ ]:
# библиотеки и загрузка данных
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
from statsmodels.stats.api import durbin_watson, acorr_breusch_godfrey # DW & LM-тесты
from statsmodels.iolib.summary2 import summary_col # вывод подгонки

from scipy.stats import chi2 # chi2-распределение и критические значения

# импорт данных
df = pd.read_csv('Tbrate.csv')

In [ ]:
# спецификация модели
mod1 = ols(formula='pi  ~ r+ y', data = df)
# подгонка исходной модели
res = mod1.fit()

#print(res.summary())

In [ ]:
# LM-тест на автокорреляцию 2-го порядка
acorr_order = 2
lm, lmpval, fval, fpval = acorr_breusch_godfrey(res, nlags=acorr_order)
print(lm.round(3)) # текстовая статистика


105.86


In [ ]:
sign_level = 0.01
# Критическое значение распределения chi2
chi2.ppf(q=1-sign_level, df=acorr_order).round(3)

np.float64(9.21)

In [ ]:
# тестовая статистика 105.859,  критическое значение 9.21 ->  есть серийная корреляция 2 порядка

In [ ]:
################### Диагностика гетероскедастичности - тест Бреуша-Пэгана/ BP test #################################

In [57]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.stats.api import het_breuschpagan, het_white # тесты на гетероскедастичность

from scipy.stats import chi2 # chi2-распределение и критические значения

# импорт данных
df = pd.read_csv('Tbrate.csv')

In [59]:
# спецификация модели
mod = ols(formula='pi  ~ r+ y', data = df)
# подгонка исходной модели
res = mod.fit()

#print(res.summary())

In [62]:
lm, lm_pvalue, fvalue, f_pvalue = het_breuschpagan(resid=res.resid, exog_het=mod.exog)
print(lm.round(3)) #тестовая статистика

5.896


In [64]:
# Задаём уровень значимости
sign_level = 0.01
# Критическое значение распределения chi2
print(chi2.ppf(q=1-sign_level, df=mod.df_model).round(3)) #критическое значение

9.21


In [ ]:
#тестовая статистика < критическое значение - нет гетероскедастичности
#тестовая статистика > критическое значение - есть гетероскедастичность

#5.896 < 9.21  - нет гетероскедастичности

In [ ]:
################### Диагностика гетероскедастичности - тест Уайта / White test #################################

In [ ]:
#предварительные шаги: загружаем данные + строим модель

In [67]:
# тест Уайта
lm, lm_pvalue, fvalue, f_pvalue = het_white(resid=res.resid, exog=mod.exog)
# LM-статистика - тестовая статистика
print(lm.round(3))

12.773


In [69]:
# Задаём уровень значимости
sign_level = 0.05
# Найдем степени свободы
k = mod.df_model # число регрессоров
df_white = 2 * k + k * (k-1) / 2
# Критическое значение распределения chi2
print(chi2.ppf(q=1-sign_level, df=df_white).round(3))

11.07


In [ ]:
#тестовая статистика < критическое значение - нет гетероскедастичности
#тестовая статистика > критическое значение - есть гетероскедастичность

#12.773 > 11.07  - есть гетероскедастичность